# URA Hackathon 2026 — EDA & Findings Report
Auto-generates every chart for the presentation into `presentation/figures/`.
Palette: #e94b27 #6c761b #ffb8a7 #324712 · white background · black text.
Run top-to-bottom in the `ura` conda env.

In [ ]:
# ===== Setup: style, paths, helpers =====
import sys, unicodedata, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

# locate project root (folder containing train_labels.csv)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "train_labels.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "train_labels.csv").exists(), "run from inside the project"
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "presentation" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT, "| figures ->", FIG)

PALETTE = ["#e94b27", "#6c761b", "#ffb8a7", "#324712"]
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
    "xtick.color": "black", "ytick.color": "black", "axes.edgecolor": "black",
    "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110,
})
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=PALETTE)
ORANGE, OLIVE, PEACH, GREEN = PALETTE

def save(fig, name):
    fig.tight_layout()
    out = FIG / (name + ".png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("saved", out.name)

def diac_frac(s):
    s = str(s)
    d = unicodedata.normalize("NFD", s)
    letters = sum(1 for c in d if c.isalpha())
    marks = sum(1 for c in d if unicodedata.category(c) == "Mn")
    return marks / letters if letters else 0.0

from scoring import cer, token_f1
from run_ocr import cache_path
labels = pd.read_csv(ROOT / "train_labels.csv", keep_default_na=False)
labels["ocr_text"] = labels["ocr_text"].fillna("")
labels["product_name"] = labels["product_name"].fillna("").str.strip()
print("train rows:", len(labels))

## 1. Product label distribution (the dominant-family concentration)

In [ ]:
nz = labels[labels.product_name != ""]["product_name"].value_counts()
top = nz.head(15)[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(top)), top.values, color=ORANGE)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index)
ax.set_xlabel("count in train"); ax.set_title("Top-15 product labels (train)")
cov = top.sum() / max(1, nz.sum())
ax.text(0.98, 0.04, f"top-15 cover {cov:.0%} of non-empty labels\n{nz.size} distinct products total",
        transform=ax.transAxes, ha="right", color=GREEN, fontsize=11)
save(fig, "01_product_longtail"); plt.show()

## 2. Empty-GT rate → the ~0.25 baseline floor

In [ ]:
p_empty = (labels.product_name == "").mean()
o_empty = (labels.ocr_text.str.strip() == "").mean()
both = ((labels.product_name == "") & (labels.ocr_text.str.strip() == "")).mean()
fig, ax = plt.subplots(figsize=(7, 5))
bars = ["product empty", "ocr empty", "both empty"]
vals = [p_empty, o_empty, both]
ax.bar(bars, vals, color=[ORANGE, OLIVE, GREEN])
for i, v in enumerate(vals): ax.text(i, v + .005, f"{v:.1%}", ha="center")
ax.set_ylabel("fraction of train"); ax.set_ylim(0, max(vals) * 1.25)
ax.set_title("Empty ground-truth rate (why all-empty already scores ~0.25)")
save(fig, "02_empty_rate"); plt.show()

## 3. OCR text length distribution (truncation at 500)

In [ ]:
L = labels.ocr_text.str.len()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(L, bins=50, color=OLIVE, edgecolor=GREEN)
ax.axvline(500, color=ORANGE, lw=2, ls="--", label="500-char GT truncation")
ax.set_xlabel("GT ocr_text length (chars)"); ax.set_ylabel("images")
ax.set_title(f"OCR text length — mean {L.mean():.0f}, median {L.median():.0f}")
ax.legend(); save(fig, "03_ocr_length"); plt.show()

## 4. Diacritic prevalence in GT (why a diacritic-preserving recognizer matters)

In [ ]:
df_d = labels[labels.ocr_text.str.strip() != ""].ocr_text.map(diac_frac)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_d, bins=40, color=PEACH, edgecolor=ORANGE)
ax.axvline(df_d.mean(), color=GREEN, lw=2, label=f"mean {df_d.mean():.2f}")
ax.set_xlabel("diacritic marks per letter (GT)"); ax.set_ylabel("images")
ax.set_title("Vietnamese diacritic density in ground-truth text")
ax.legend(); save(fig, "04_diacritic_density"); plt.show()

## 5. OCR engine bake-off — CER & diacritic preservation
The metric's CER term is **diacritic-sensitive**, so an engine that strips diacritics
is structurally capped. Compared on the common train images with ground truth.

In [ ]:
ENGINES = {"PaddleOCR": "paddleocr", "VietOCR (base)": "vietocr", "VietOCR-FT (ours)": "vietocr_ft"}
gt = labels.set_index("image_id")["ocr_text"]
series = {}
for nm, eng in ENGINES.items():
    f = cache_path(eng, "all")
    if Path(f).exists():
        s = pd.read_parquet(f)[["image_id", "ocr_text"]].set_index("image_id")["ocr_text"].fillna("")
        series[nm] = s
common = set(gt.index)
for s in series.values(): common &= set(s.index)
common = sorted(common)
rows = []
for nm, s in series.items():
    cers = [cer(gt[i], s.get(i, "")) for i in common]
    rows.append({"engine": nm, "CER": float(np.mean(cers)), "ocr_term": 1 - float(np.mean(cers)),
                 "diacritic/letter": float(np.mean([diac_frac(s.get(i, "")) for i in common]))})
estats = pd.DataFrame(rows); print(estats.to_string(index=False), "| n =", len(common))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(estats.engine, estats.CER, color=[ORANGE, OLIVE, GREEN])
for i, v in enumerate(estats.CER): axes[0].text(i, v + .005, f"{v:.3f}", ha="center")
axes[0].set_ylabel("mean CER (lower=better)"); axes[0].set_title("CER by OCR engine")
axes[1].bar(estats.engine, estats["diacritic/letter"], color=[ORANGE, OLIVE, GREEN])
for i, v in enumerate(estats["diacritic/letter"]): axes[1].text(i, v + .005, f"{v:.2f}", ha="center")
axes[1].set_ylabel("diacritic marks / letter"); axes[1].set_title("Diacritic preservation (GT ref ~%.2f)" % df_d.mean())
for a in axes: a.tick_params(axis="x", rotation=15)
save(fig, "05_engine_bakeoff"); plt.show()

## 6. Where OCR error concentrates — CER bands (VietOCR-FT vs GT)

In [ ]:
ft = pd.read_parquet(cache_path("vietocr_ft", "all"))[["image_id", "ocr_text"]].rename(columns={"ocr_text": "pred"})
m = labels.merge(ft, on="image_id"); m["pred"] = m["pred"].fillna("")
m = m[m.ocr_text.str.strip() != ""]
m["cer"] = [cer(g, p) for g, p in zip(m.ocr_text, m.pred)]
bands = [("[0,0.1)", 0, .1), ("[0.1,0.3)", .1, .3), ("[0.3,0.6)", .3, .6), ("[0.6,1.0]", .6, 1.01)]
counts = [((m.cer >= lo) & (m.cer < hi)).mean() for _, lo, hi in bands]
fig, ax = plt.subplots(figsize=(8, 5))
cols = [GREEN, OLIVE, PEACH, ORANGE]
ax.bar([b[0] for b in bands], counts, color=cols)
for i, v in enumerate(counts): ax.text(i, v + .005, f"{v:.0%}", ha="center")
ax.set_ylabel("fraction of text-bearing images"); ax.set_title("CER distribution — the ~27% total-failure band (region mismatch)")
save(fig, "06_cer_bands"); plt.show()

## 7. Oracle vs real product F1 — the OCR-noise penalty
Product head scored with **perfect (GT) text** vs **real OCR** input. The gap is the
headroom gated by OCR quality. (In-sample fit, illustrative of the decomposition.)

In [ ]:
from product_calibrated import CalibratedRuleHead
head = CalibratedRuleHead(use_classifier_fallback=True, min_pprod=0.55, gate_threshold=0.75).fit(
    labels[["image_id", "ocr_text", "product_name"]])
oracle = head.predict_batch(labels["ocr_text"])
f1_oracle = float(np.mean([token_f1(g, p) for g, p in zip(labels.product_name, oracle)]))
mo = labels.merge(ft, on="image_id"); mo["pred"] = mo["pred"].fillna("")
real = head.predict_batch(mo["pred"])
f1_real = float(np.mean([token_f1(g, p) for g, p in zip(mo.product_name, real)]))
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["oracle (GT text)", "real (VietOCR-FT)"], [f1_oracle, f1_real], color=[GREEN, ORANGE])
ax.annotate("", xy=(1, f1_real), xytext=(1, f1_oracle), arrowprops=dict(arrowstyle="<->", color="black"))
ax.text(1.06, (f1_oracle + f1_real) / 2, f"OCR-noise\npenalty\n{f1_oracle - f1_real:.3f}", va="center", color=ORANGE)
for i, v in enumerate([f1_oracle, f1_real]): ax.text(i, v + .008, f"{v:.3f}", ha="center")
ax.set_ylabel("product token-F1"); ax.set_ylim(0, max(f1_oracle, f1_real) * 1.2)
ax.set_title("Product head: oracle vs real-OCR F1")
save(fig, "07_oracle_vs_real"); plt.show()

## 8. Score progression (public leaderboard, documented results)

In [ ]:
# documented public-LB results from our runs
prog = [("v8 classifier", 0.6232), ("v10 calibrated\n(ours)", 0.6685), ("our OCR +\nteammate product", 0.6959)]
names = [p[0] for p in prog]; vals = [p[1] for p in prog]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(vals)), vals, "-o", color=ORANGE, lw=2.5, ms=9)
for i, v in enumerate(vals): ax.text(i, v + .003, f"{v:.4f}", ha="center", color=GREEN)
ax.axhline(0.6495, color=OLIVE, ls="--", lw=1.5, label="0.6495 rival (reference)")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
ax.set_ylabel("public LB score"); ax.set_title("Public-leaderboard progression")
ax.legend(); save(fig, "08_score_progression"); plt.show()

## 9. The experiment log — every idea, judged by one fixed CV
Composite-score deltas vs the calibrated baseline. Green = kept/neutral, red = rejected.
F1-only and ocr_term-only deltas converted to composite (×0.6 / ×0.4).

In [ ]:
# documented CV deltas (composite units)
exps = [
    ("prep+beam OCR", +0.0024), ("highlands-first reorder", +0.0005),
    ("friend canonicalizer", -0.0025), ("aggressive multi-scale detect", -0.0164),
    ("friend evidence_override", -0.0226), ("friend full head", -0.0342),
    ("diacritic restoration", -0.0440), ("news-context abstention", -0.0599),
]
exps = sorted(exps, key=lambda x: x[1])
names = [e[0] for e in exps]; dels = [e[1] for e in exps]
cols = [OLIVE if d >= 0 else ORANGE for d in dels]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(names)), dels, color=cols)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.axvline(0, color="black", lw=1)
for i, d in enumerate(dels): ax.text(d + (0.001 if d >= 0 else -0.001), i, f"{d:+.4f}", va="center",
                                     ha="left" if d >= 0 else "right", fontsize=10)
ax.set_xlabel("CV composite delta vs baseline"); ax.set_title("Disciplined iteration: one change, one CV verdict")
save(fig, "09_experiment_log"); plt.show()

## 10. Phase-1 vs Phase-2 distribution shift (dominant-family concentration)

In [ ]:
import re
def fold(s):
    s = unicodedata.normalize("NFD", str(s).lower()).replace(chr(273), "d")
    return re.sub(r"[^a-z0-9 ]", " ", s)
MARK = r"canfoco|cot den|\bnan\b|nestle|highland|pate|ha long|do hop"
res = {}
for label, eng, split in [("Phase 1 (public test)", "vietocr_ft", "test"),
                          ("Phase 2 (private test)", "vietocr_ft_phase2test", None)]:
    f = cache_path(eng, "test") if split else (ROOT / "cache" / "ocr_vietocr_ft_phase2test.parquet")
    if Path(f).exists():
        s = pd.read_parquet(f)["ocr_text"].fillna("").map(fold)
        res[label] = s.str.contains(MARK, regex=True).mean()
fig, ax = plt.subplots(figsize=(7, 5))
ks = list(res); vs = [res[k] for k in ks]
ax.bar(ks, vs, color=[ORANGE, GREEN])
for i, v in enumerate(vs): ax.text(i, v + .005, f"{v:.0%}", ha="center")
ax.set_ylabel("OCR rows containing a dominant-family marker")
ax.set_title("Distribution shift: Phase 2 is more diverse")
save(fig, "10_phase_shift"); plt.show()
print("\nAll figures saved to", FIG)